# CNNs avec PyTorch Lightning sur des données de paysage

Dans ces travaux pratiques, nous classons les photos du jeu des paysages (Intel Image Classification, 6 classes) :

1. avec le CNN des slides, puis un CNN plus profond, que vous écrivez ;
2. en augmentant les données avec `torchvision.transforms.v2` ;
3. avec trois architectures historiques : LeNet, AlexNet et un bloc Inception ;
4. en analysant les erreurs sur l'ensemble de test ;
5. par transfert d'apprentissage : EfficientNetV2-B0 pré-entraîné sur ImageNet (bibliothèque `timm`), gelé puis fine-tuné (partie « Transfert d'apprentissage ») ;
6. facultatif : avec un *Vision Transformer* pré-entraîné (partie « Transfert d'apprentissage avec des transformeurs »).

La partie 5 va avec le chapitre « Transfer learning » des slides : arrêtez-vous avant elle si vous n'en êtes qu'au chapitre sur les CNN. La partie 6 est facultative : elle reprend la démarche de la partie 5 avec une architecture, le *Vision Transformer*, que la partie résume en quelques lignes et que les slides présentent dans un chapitre à part (« Transformers »), s'il est au programme.

## Vérification de l'utilisation de GPU

Allez dans le menu `Exécution > Modifier le type d'exécution` et vérifiez que l'accélérateur matériel est un GPU : sans lui, chaque entraînement de ce TP prendrait des heures.

In [ ]:
!nvidia-smi

## Téléchargement du jeu des paysages

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh landscape
!ls landscape
print("***")
!ls -l landscape/seg_train
print("***")
!ls -l landscape/seg_pred

## Installation et imports

In [ ]:
!pip install -q lightning torchmetrics timm

In [ ]:
import itertools
import pathlib

import lightning
import matplotlib.pyplot as plt
import numpy
import pandas
import PIL.Image
import seaborn
import sklearn.metrics
import timm
import torch
import torchmetrics
import tqdm.notebook
import transformers
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

# Mêmes tirages aléatoires à chaque exécution (PyTorch, NumPy, Python) :
# des résultats reproductibles
lightning.seed_everything(0)

## Préparation des données

Les slides (« Charger les images ») lisent les images avec `ImageFolder`, qui les relit sur le disque à chaque époque. Ce TP les entraîne une vingtaine de fois, à trois tailles : il les charge une fois pour toutes en mémoire.

`charge_images` lit les images d'un dossier, sous-dossier par sous-dossier : l'étiquette d'une image est l'indice du nom de son sous-dossier dans `noms_etiquettes` (0 = `buildings`, …, 5 = `street`). Elle les redimensionne à `taille` × `taille` pixels et les garde toutes en mémoire, dans un tenseur d'entiers de 0 à 255, d'un octet chacun. Les 14 034 images d'entraînement y prennent 0,95 Go en 150 × 150, puis 1,6 Go en 192 × 192 et 2,1 Go en 224 × 224, les tailles du transfert d'apprentissage : la RAM de Colab (12,7 Go) les contient.

`DatasetPaysages` ne convertit une image en flottants dans [0, 1] qu'au moment de la lire, batch par batch : en flottants de 4 octets, le jeu entier prendrait quatre fois plus de place. Il lui applique aussi, pour l'entraînement, une transformation d'augmentation facultative (partie « Augmentation de données »).

`fabrique_chargeurs` garde 20 % des images d'entraînement pour la validation (2 806 images, comme le `random_split` des slides), et `fabrique_chargeur` sert pour l'ensemble de test.

In [ ]:
FORME_ENTREE = (3, 150, 150)


noms_etiquettes = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
indice_etiquette = {l: i for i, l in enumerate(noms_etiquettes)}


def charge_images(
  chemin_dossier: pathlib.Path,
  taille: int = FORME_ENTREE[1],
  melange: bool = True,
  cree_etiquettes: bool = True,
) -> tuple[torch.Tensor, torch.Tensor] | torch.Tensor:
  images = []
  if cree_etiquettes:
    etiquettes = []

  # On parcourt les sous-dossiers de la racine : chacun correspond à une classe
  for chemin_sous_dossier in tqdm.notebook.tqdm(
    list(chemin_dossier.iterdir()), desc="Traitement des dossiers"
  ):
    nom_dossier = chemin_sous_dossier.name

    if cree_etiquettes:
      # L'étiquette est l'indice du nom du dossier
      etiquette = indice_etiquette.get(nom_dossier)

    # On ajoute chaque image du dossier courant à notre jeu de données
    for chemin_image in tqdm.notebook.tqdm(
      list(chemin_sous_dossier.iterdir()), desc=f"Dossier {nom_dossier}", leave=False
    ):
      # Pillow charge l'image, en RVB, à la taille demandée
      images.append(
        numpy.array(
          PIL.Image.open(chemin_image).convert("RGB").resize((taille, taille))
        )
      )
      if cree_etiquettes:
        etiquettes.append(etiquette)

  # PyTorch attend les canaux en deuxième position : on passe donc de la forme
  # (N, H, W, C) à la forme (N, C, H, W)
  images = torch.from_numpy(numpy.array(images)).permute(0, 3, 1, 2).contiguous()
  if cree_etiquettes:
    etiquettes = torch.tensor(etiquettes)

  if melange:
    permutation = torch.randperm(images.shape[0])
    images = images[permutation]
    if cree_etiquettes:
      etiquettes = etiquettes[permutation]

  if cree_etiquettes:
    return images, etiquettes
  else:
    return images

In [ ]:
class DatasetPaysages(Dataset):
  """Dataset d'images, converties en flottants dans [0, 1] à la volée.

  `augmentation`, une transformation de torchvision (pour l'entraînement
  seulement), s'applique à chaque lecture d'une image : chaque époque en voit
  une variante différente.
  """

  def __init__(
    self,
    images: torch.Tensor,
    etiquettes: torch.Tensor | None = None,
    augmentation: v2.Transform | None = None,
  ) -> None:
    self.images = images
    self.etiquettes = etiquettes
    self.augmentation = augmentation

  def __len__(self) -> int:
    return self.images.shape[0]

  def __getitem__(self, index: int) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
    image = self.images[index]
    if self.augmentation is not None:
      image = self.augmentation(image)
    image = image.float() / 255.0
    if self.etiquettes is None:
      return image
    return image, self.etiquettes[index]


def fabrique_chargeur(
  images: torch.Tensor,
  etiquettes: torch.Tensor | None = None,
  taille_lot: int = 32,
  melange: bool = False,
  augmentation: v2.Transform | None = None,
) -> DataLoader:
  # Les images sont déjà en mémoire : le processus principal lit lui-même les
  # lots, sans démarrer de processus pour chacun des entraînements du TP
  return DataLoader(
    DatasetPaysages(images, etiquettes, augmentation),
    batch_size=taille_lot,
    shuffle=melange,
  )


def fabrique_chargeurs(
  images: torch.Tensor,
  etiquettes: torch.Tensor,
  taille_lot: int = 32,
  fraction_validation: float = 0.2,
  augmentation: v2.Transform | None = None,
) -> tuple[DataLoader, DataLoader]:
  """Sépare les données entre entraînement et validation.

  L'augmentation ne s'applique qu'aux images d'entraînement.
  """
  nb_validation = int(images.shape[0] * fraction_validation)
  nb_entrainement = images.shape[0] - nb_validation
  return (
    fabrique_chargeur(
      images[:nb_entrainement],
      etiquettes[:nb_entrainement],
      taille_lot,
      melange=True,
      augmentation=augmentation,
    ),
    fabrique_chargeur(
      images[nb_entrainement:], etiquettes[nb_entrainement:], taille_lot
    ),
  )


def rend_affichable(image: torch.Tensor) -> numpy.ndarray:
  """Passe d'un tenseur (C, H, W) à un tableau (H, W, C) affichable."""
  return image.permute(1, 2, 0).numpy()

## Regardons les données

In [ ]:
images, etiquettes = charge_images(pathlib.Path("landscape") / "seg_train")

In [ ]:
print(f"Forme des images : {tuple(images.shape)}")
print(f"Forme des étiquettes : {tuple(etiquettes.shape)}")

effectifs = torch.bincount(etiquettes, minlength=len(noms_etiquettes))
plt.bar(noms_etiquettes, effectifs.numpy())
plt.title("Nombre d'images par classe (entraînement et validation)")
plt.ylabel("Nombre d'images")
plt.xlabel("Classe")
plt.show()

In [ ]:
# 25 images tirées au hasard, sans remise (pas deux fois la même image)
indices_aleatoires = numpy.random.choice(images.shape[0], size=(5, 5), replace=False)

# figsize agrandit la figure, petite par défaut
_, ax = plt.subplots(5, 5, figsize=(15, 15))
for i, j in itertools.product(range(5), repeat=2):
  indice_image = indices_aleatoires[i, j]
  ax[i, j].imshow(rend_affichable(images[indice_image]))
  ax[i, j].set_title(
    f"Image {indice_image} ({noms_etiquettes[int(etiquettes[indice_image])]})"
  )
  ax[i, j].axis("off")
plt.show()

## Création du modèle

`ClassifieurImages` enveloppe chacun de nos modèles dans un `LightningModule` : la perte d'entropie croisée, l'optimiseur AdamW et la justesse (*accuracy*) en entraînement, validation et test. `fabrique_trainer` crée le `Trainer` qui entraîne puis évalue un modèle, en écrivant ses métriques dans un fichier CSV.

Le premier modèle est le CNN des slides (« Définir un CNN ») : deux blocs convolution + ReLU + pooling, puis un pooling global (`AdaptiveAvgPool2d(1)`) qui garde une valeur par canal, et une couche dense de sortie, un logit par classe.

In [ ]:
class ClassifieurImages(lightning.LightningModule):
  """Enveloppe Lightning commune à tous nos modèles de classification.

  Le modèle encapsulé renvoie des logits : le softmax est appliqué par la
  fonction de perte pendant l'apprentissage, et explicitement au moment de la
  prédiction.
  """

  def __init__(
    self,
    modele: nn.Module,
    taux_apprentissage: float = 1e-4,
    decroissance_poids: float = 0.0,
    nb_classes_sortie: int = len(noms_etiquettes),
    taille_entree: int = FORME_ENTREE[1],
  ) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["modele"])
    self.modele = modele
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(
      1, FORME_ENTREE[0], taille_entree, taille_entree
    )
    # torchmetrics demande une instance de métrique par étape
    self.justesses = nn.ModuleDict(
      {
        f"justesse_{etape}": torchmetrics.Accuracy(
          task="multiclass", num_classes=nb_classes_sortie
        )
        for etape in ("entrainement", "val", "test")
      }
    )

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.modele(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], etape: str) -> torch.Tensor:
    images, etiquettes = batch
    logits = self(images)
    perte = nn.functional.cross_entropy(logits, etiquettes)
    justesse = self.justesses[f"justesse_{etape}"]
    justesse(logits, etiquettes)
    self.log(f"perte_{etape}", perte, on_step=False, on_epoch=True, prog_bar=True)
    self.log(f"justesse_{etape}", justesse, on_step=False, on_epoch=True, prog_bar=True)
    return perte

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "entrainement")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "test")

  def predict_step(
    self, batch: torch.Tensor | tuple[torch.Tensor, ...], indice_lot: int
  ) -> torch.Tensor:
    images = batch[0] if isinstance(batch, (list, tuple)) else batch
    return self(images).softmax(dim=-1).cpu()

  def configure_optimizers(self) -> torch.optim.Optimizer:
    # AdamW applique la weight decay directement aux poids (voir les slides) ;
    # les paramètres gelés (requires_grad=False) n'ont pas de gradient et sont
    # ignorés par l'optimiseur
    return torch.optim.AdamW(
      self.parameters(),
      lr=self.hparams.taux_apprentissage,
      weight_decay=self.hparams.decroissance_poids,
    )


def fabrique_trainer(nb_epoques_max: int, nom: str) -> lightning.Trainer:
  """Crée un Trainer qui journalise ses métriques dans un fichier CSV."""
  return lightning.Trainer(
    max_epochs=nb_epoques_max,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("journaux", name=nom),
    # La barre de progression de tqdm, plutôt que celle de rich que Lightning
    # choisit quand rich est installé (Colab) : elle a figé des entraînements
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
    num_sanity_val_steps=0,
    log_every_n_steps=10,
  )

In [ ]:
# Le CNN des slides
modele = nn.Sequential(
  nn.Conv2d(3, 32, kernel_size=3),  # 32 filtres 3 × 3
  nn.ReLU(),
  nn.MaxPool2d(2),  # fenêtres 2 × 2, pas de 2
  nn.Conv2d(32, 64, kernel_size=3),
  nn.ReLU(),
  nn.MaxPool2d(2),
  nn.AdaptiveAvgPool2d(1),  # une valeur par canal
  nn.Flatten(),  # (N, 64, 1, 1) → (N, 64)
  nn.Linear(64, 6),  # un logit par classe, softmax dans la perte
)

# L'enveloppe Lightning définit la fonction de perte et l'optimiseur
classifieur = ClassifieurImages(modele, taux_apprentissage=1e-3)

# Résumé du modèle : chaque couche, ses paramètres, ses formes d'entrée et de sortie
print(ModelSummary(classifieur, max_depth=-1))

## Pouvez-vous expliquer les différents nombres de paramètres ?

Retrouvez par le calcul le nombre de paramètres de chaque couche du résumé. Pourquoi certaines couches n'en ont-elles aucun ?

### Solution

- `modele.0` : chacun des 32 filtres a 3 × 3 poids par canal d'entrée, sur 3 canaux (RVB), plus un biais : (3 × 3 × 3 + 1) × 32 = 896.
- `modele.3` : 64 filtres, qui couvrent chacun les 32 canaux de la couche précédente : (3 × 3 × 32 + 1) × 64 = 18 496.
- `modele.8` : 64 entrées (une par canal, après le pooling global) et 6 sorties, plus 6 biais : 64 × 6 + 6 = 390.
- Les ReLU, les poolings, le pooling global et `Flatten` n'apprennent rien : ils appliquent une opération fixe (annuler les valeurs négatives, prendre un maximum ou une moyenne, aplatir). Les métriques de justesse (`justesses`) n'ont pas de paramètres non plus.

Total : 19 782 paramètres, comme le CNN Keras des slides. Le pooling global garde la tête petite : sans lui, `Flatten` passerait à la couche dense 64 × 36 × 36 = 82 944 valeurs, soit 497 670 paramètres.

## Apprentissage

La cellule suivante définit deux fonctions pour la suite :

- `trace_mesures(trainer)` trace les courbes de justesse et de perte d'un entraînement ;
- `entraine(modele, nb_epoques_max, nom, taux_apprentissage)` regroupe les étapes d'un entraînement : l'enveloppe, la séparation des données par `fabrique_chargeurs` (batchs de 32), le `Trainer` de `fabrique_trainer`, `trainer.fit`, qui entraîne le modèle en l'évaluant sur la validation à la fin de chaque époque, puis les courbes. Elle renvoie l'enveloppe entraînée et son `Trainer`, qui sert ensuite à l'évaluer.

Elle entraîne ensuite le CNN des slides comme dans les slides (« Entraîner et tester le modèle ») : 10 époques, avec un taux d'apprentissage de 1e-3.

In [ ]:
def trace_mesures(trainer: lightning.Trainer) -> None:
  """Trace la justesse et la perte, en entraînement et en validation."""
  # Les métriques d'entraînement et de validation ne sont pas écrites sur les
  # mêmes lignes du fichier CSV : on les regroupe par époque
  mesures = (
    pandas.read_csv(pathlib.Path(trainer.logger.log_dir) / "metrics.csv")
    .groupby("epoch")
    .mean(numeric_only=True)
    .dropna(subset=["perte_entrainement", "perte_val"])
  )
  for mesure, nom in (("justesse", "Justesse"), ("perte", "Perte")):
    plt.plot(mesures.index, mesures[f"{mesure}_entrainement"], label="Entraînement")
    plt.plot(mesures.index, mesures[f"{mesure}_val"], label="Validation")
    plt.title(f"{nom} du modèle")
    plt.ylabel(nom)
    plt.xlabel("Époque")
    plt.legend()
    plt.show()


def entraine(
  modele: nn.Module,
  nb_epoques_max: int,
  nom: str,
  taux_apprentissage: float = 1e-4,
  augmentation: v2.Transform | None = None,
) -> tuple[ClassifieurImages, lightning.Trainer]:
  """Entraîne un modèle sur les images 150 × 150, puis trace ses courbes.

  Renvoie l'enveloppe entraînée et le Trainer, qui sert ensuite à l'évaluer.
  """
  classifieur = ClassifieurImages(modele, taux_apprentissage=taux_apprentissage)
  print(ModelSummary(classifieur, max_depth=-1))
  chargeur_entrainement, chargeur_validation = fabrique_chargeurs(
    images, etiquettes, augmentation=augmentation
  )
  trainer = fabrique_trainer(nb_epoques_max, nom)
  trainer.fit(classifieur, chargeur_entrainement, chargeur_validation)
  trace_mesures(trainer)
  return classifieur, trainer


classifieur, trainer = entraine(
  modele, nb_epoques_max=10, nom="cnn_des_slides", taux_apprentissage=1e-3
)

## Améliorez cette performance

Le CNN des slides plafonne vers 65 % de justesse : deux convolutions ne voient que de petits motifs, trop peu pour reconnaître une scène. Écrivez une fonction `cnn_profond()` qui renvoie un CNN plus profond, avec des poids neufs à chaque appel, puis entraînez-le avec `entraine` :

- des blocs convolution + ReLU + pooling répétés, jusqu'à des cartes de quelques pixels de côté ;
- du *dropout* avant les couches denses de la tête, contre le surapprentissage ;
- entre 10 et 20 époques, chacune en moins d'une minute sur le GPU de Colab.

Objectif : plus de 70 % de justesse en validation. La solution atteint environ 85 % en 15 époques, de 26 secondes chacune sur une RTX A4000. `cnn_profond()` resservira pour l'augmentation de données.

**Documentation utile** :

- https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html
- https://docs.pytorch.org/docs/stable/nn.init.html#torch.nn.init.orthogonal_
- https://docs.pytorch.org/docs/stable/nn.init.html#torch.nn.init.xavier_uniform_

In [ ]:
def cnn_profond() -> nn.Sequential: ...


# Votre code ici

### Conseils supplémentaires

- Un bloc : `nn.Conv2d(canaux_entree, 200, kernel_size=3, padding="same")`, `nn.ReLU()`, puis `nn.MaxPool2d(2, ceil_mode=True)`. La première convolution prend les 3 canaux de l'image, les suivantes les 200 de la précédente.
- Avec `padding="same"`, la convolution garde la taille des cartes et seul le pooling la divise par 2, arrondie au-dessus avec `ceil_mode=True` : 150 → 75 → 38 → 19 → 10 → 5 → 3 en 6 blocs. Une 7ᵉ convolution, sans pooling, termine sur 200 cartes 3 × 3.
- La solution initialise les poids de chaque convolution avec `nn.init.orthogonal_(convolution.weight)`, et ceux des couches denses avec `nn.init.xavier_uniform_` (l'initialisation de Glorot, celle de Keras), leurs biais à zéro : avec l'initialisation par défaut de PyTorch, plus petite, le signal diminue à chaque couche de la tête et le modèle apprend plus lentement.
- Tête : `nn.Flatten()`, qui donne 200 × 3 × 3 = 1 800 valeurs, puis trois couches `nn.Linear` de 200, 100 et 50 neurones suivies chacune d'une ReLU, et la sortie `nn.Linear(50, 6)`. Chacune de ces quatre couches est précédée de `nn.Dropout(0.2)`.
- Entraînement : `entraine(cnn_profond(), nb_epoques_max=15, nom="cnn_profond")`, avec le taux d'apprentissage par défaut de `entraine` (1e-4).

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def bloc_convolution(canaux_entree: int) -> list[nn.Module]:
  convolution = nn.Conv2d(canaux_entree, 200, kernel_size=3, padding="same")
  nn.init.orthogonal_(convolution.weight)
  return [convolution, nn.ReLU()]


def cnn_profond() -> nn.Sequential:
  blocs = []
  canaux_entree = 3
  for _ in range(6):
    blocs += [*bloc_convolution(canaux_entree), nn.MaxPool2d(...)]
    canaux_entree = 200
  modele = nn.Sequential(
    *blocs,
    *bloc_convolution(200),
    nn.Flatten(),
    nn.Dropout(0.2),
    nn.Linear(..., 200),
    nn.ReLU(),
    ...,
    nn.Linear(50, 6),
  )
  for couche in modele:
    if isinstance(couche, nn.Linear):
      nn.init.xavier_uniform_(couche.weight)
      nn.init.zeros_(couche.bias)
  return modele


classifieur, trainer = entraine(cnn_profond(), nb_epoques_max=..., nom="cnn_profond")
```

### Solution

In [ ]:
def bloc_convolution(canaux_entree: int) -> list[nn.Module]:
  """Convolution 3 × 3 à 200 filtres, qui garde la taille des cartes, puis ReLU."""
  convolution = nn.Conv2d(canaux_entree, 200, kernel_size=3, padding="same")
  nn.init.orthogonal_(convolution.weight)
  return [convolution, nn.ReLU()]


def cnn_profond() -> nn.Sequential:
  """Le CNN profond de la solution, avec des poids neufs à chaque appel."""
  blocs = []
  canaux_entree = 3
  for _ in range(6):  # cartes de 150 → 75 → 38 → 19 → 10 → 5 → 3 pixels de côté
    blocs += [*bloc_convolution(canaux_entree), nn.MaxPool2d(2, ceil_mode=True)]
    canaux_entree = 200
  modele = nn.Sequential(
    *blocs,
    *bloc_convolution(200),
    nn.Flatten(),  # 200 cartes 3 × 3 → 1 800 valeurs
    nn.Dropout(0.2),
    nn.Linear(200 * 3 * 3, 200),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(200, 100),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(50, 6),
  )
  # Initialisation de Glorot pour les couches denses, celle de Keras : avec
  # celle de PyTorch, plus petite, le signal diminue à chacune des quatre
  # couches de la tête, et le modèle apprend plus lentement
  for couche in modele:
    if isinstance(couche, nn.Linear):
      nn.init.xavier_uniform_(couche.weight)
      nn.init.zeros_(couche.bias)
  return modele

In [ ]:
classifieur, trainer = entraine(cnn_profond(), nb_epoques_max=15, nom="cnn_profond")

## Augmentation de données

Quand la justesse d'entraînement continue de monter alors que celle de validation stagne, le modèle surapprend. L'augmentation de données (slide « Transformations et augmentation de données ») lui présente à chaque époque des variantes des images d'entraînement.

Choisissez deux ou trois transformations de `torchvision.transforms.v2` qui gardent la classe d'un paysage, avec leur amplitude : lesquelles ne la gardent pas (un miroir vertical ? une rotation de 90° ? un recadrage sur un dixième de l'image ?) ? Composez-les en une transformation `augmentation`. Passée à `entraine(..., augmentation=augmentation)`, elle ne s'applique qu'aux images d'entraînement : celles de validation restent intactes.

Entraînez la même architecture, avec des poids neufs (`cnn_profond()`), pendant 15 époques, et comparez l'écart entre les courbes d'entraînement et de validation à celui du CNN profond sans augmentation.

**Documentation utile** :

- https://docs.pytorch.org/vision/stable/transforms.html#v2-api-reference-recommended

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `v2.Compose([...])` enchaîne des transformations, dans l'ordre de la liste.
- Des transformations qui gardent la classe d'un paysage, et leurs paramètres :
  - `v2.RandomHorizontalFlip()` : miroir gauche-droite, avec une probabilité de 0,5 par défaut ;
  - `v2.RandomResizedCrop(150, scale=(0.7, 1.0), antialias=True)` : recadrage sur 70 à 100 % de la surface, ramené à 150 × 150 pixels ;
  - `v2.RandomRotation(10)` : rotation d'au plus 10° dans un sens ou dans l'autre ;
  - `v2.RandomAffine(degrees=0, translate=(0.1, 0.1))` : décalage d'au plus 10 % de la largeur et de la hauteur ;
  - `v2.ColorJitter(brightness=0.2, contrast=0.2)` : luminosité et contraste multipliés par un facteur entre 0,8 et 1,2.
- À éviter : `v2.RandomVerticalFlip()` (le ciel passerait en bas), de grandes rotations, un recadrage si petit qu'il ne garde qu'un bout de ciel ou de mur, un changement de teinte (`hue`) qui rendrait la mer verte ou un glacier jaune.
- Les transformations travaillent sur les entiers de 0 à 255 des images, que `DatasetPaysages` convertit ensuite en flottants.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
augmentation = v2.Compose([..., ...])
classifieur, trainer = entraine(
  cnn_profond(), nb_epoques_max=15, nom="cnn_profond_augmente", augmentation=...
)
```

### Solution

In [ ]:
augmentation = v2.Compose(
  [
    # recadrage sur 70 à 100 % de la surface, ramené à 150 × 150
    v2.RandomResizedCrop(150, scale=(0.7, 1.0), antialias=True),
    v2.RandomHorizontalFlip(),  # miroir, une image sur deux
  ]
)
classifieur, trainer = entraine(
  cnn_profond(),
  nb_epoques_max=15,
  nom="cnn_profond_augmente",
  augmentation=augmentation,
)

Une solution parmi d'autres : le recadrage aléatoire et le miroir horizontal. Chaque époque voit des variantes différentes des images : la justesse d'entraînement monte moins vite (85 % environ après 15 époques, contre 91 % sans augmentation) et reste sous celle de validation presque jusqu'au bout, alors que sans augmentation elle la dépassait dès la 7ᵉ époque environ. La validation finit vers 85 % dans les deux cas, mais l'écart entre les deux courbes a disparu. Le modèle doit apprendre des caractéristiques qui résistent au cadrage et à l'orientation, au lieu de mémoriser les images d'entraînement. L'évaluation sur le test et l'analyse d'erreur portent sur ce modèle : `classifieur` et son `trainer`.

## Architectures historiques

Trois architectures du chapitre sur les CNN, à entraîner puis comparer au CNN profond : leur code est donné. Chacune a ses propres noms de variables, et `classifieur` et `trainer` restent ceux du CNN profond augmenté.

### LeNet

[LeNet-5](https://en.wikipedia.org/wiki/LeNet) (1998), conçu pour des chiffres manuscrits de 32 × 32 pixels : des convolutions 5 × 5, des activations sigmoïdes et des poolings par moyenne.

In [ ]:
le_net = nn.Sequential(
  nn.Conv2d(3, 6, kernel_size=5, padding="same"),
  nn.Sigmoid(),
  nn.AvgPool2d(2),  # 150 → 75
  nn.Conv2d(6, 16, kernel_size=5),  # 75 → 71
  nn.Sigmoid(),
  nn.AvgPool2d(2),  # 71 → 35
  nn.Flatten(),
  nn.Linear(16 * 35 * 35, 120),
  nn.Sigmoid(),
  nn.Linear(120, 84),
  nn.Sigmoid(),
  nn.Linear(84, 6),
)
classifieur_le_net, trainer_le_net = entraine(le_net, nb_epoques_max=15, nom="le_net")

### AlexNet

[AlexNet](https://en.wikipedia.org/wiki/AlexNet) (2012), vainqueur d'ImageNet : une première convolution 11 × 11 de pas 4, des ReLU, du *dropout* dans une tête de deux couches de 4 096 neurones.

In [ ]:
alex_net = nn.Sequential(
  nn.Conv2d(3, 96, kernel_size=11, stride=4),  # 150 → 35
  nn.ReLU(),
  nn.MaxPool2d(3, stride=2),  # 35 → 17
  nn.Conv2d(96, 256, kernel_size=5, padding="same"),
  nn.ReLU(),
  nn.MaxPool2d(3, stride=2),  # 17 → 8
  nn.Conv2d(256, 384, kernel_size=3, padding="same"),
  nn.ReLU(),
  nn.Conv2d(384, 384, kernel_size=3, padding="same"),
  nn.ReLU(),
  nn.Conv2d(384, 256, kernel_size=3, padding="same"),
  nn.ReLU(),
  nn.MaxPool2d(3, stride=2),  # 8 → 3
  nn.Flatten(),
  nn.Linear(256 * 3 * 3, 4096),
  nn.ReLU(),
  nn.Dropout(0.5),
  nn.Linear(4096, 4096),
  nn.ReLU(),
  nn.Dropout(0.5),
  nn.Linear(4096, 6),
)
classifieur_alex_net, trainer_alex_net = entraine(
  alex_net, nb_epoques_max=15, nom="alex_net"
)

### Bloc Inception

Un bloc [Inception](https://en.wikipedia.org/wiki/Inception_(deep_learning_architecture)) (GoogLeNet, 2014) : quatre branches en parallèle (convolutions 1 × 1, 3 × 3, 5 × 5 et pooling), concaténées sur l'axe des canaux. Un réseau complet en empile plusieurs. Le flux de données n'est plus séquentiel : le bloc est un [module PyTorch](https://docs.pytorch.org/docs/stable/notes/modules.html) personnalisé, dont la méthode `forward` combine les branches.

In [ ]:
class BlocInception(nn.Module):
  """Bloc Inception : quatre branches concaténées sur l'axe des canaux."""

  def __init__(self, canaux_entree: int, canaux_sortie: int = 64) -> None:
    super().__init__()

    # conv 1 × 1
    self.conv11 = nn.Sequential(nn.Conv2d(canaux_entree, canaux_sortie, 1), nn.ReLU())

    # conv 1 × 1 puis conv 3 × 3
    self.conv33 = nn.Sequential(
      nn.Conv2d(canaux_entree, canaux_sortie, 1),
      nn.ReLU(),
      nn.Conv2d(canaux_sortie, canaux_sortie, 3, padding="same"),
      nn.ReLU(),
    )

    # conv 1 × 1 puis conv 5 × 5
    self.conv55 = nn.Sequential(
      nn.Conv2d(canaux_entree, canaux_sortie, 1),
      nn.ReLU(),
      nn.Conv2d(canaux_sortie, canaux_sortie, 5, padding="same"),
      nn.ReLU(),
    )

    # max pooling 3 × 3 de pas 1 puis conv 1 × 1
    self.pooling_max = nn.Sequential(
      nn.MaxPool2d(3, 1, padding=1),
      nn.Conv2d(canaux_entree, canaux_sortie, 1),
      nn.ReLU(),
    )

  def forward(self, entrees: torch.Tensor) -> torch.Tensor:
    # concaténation sur l'axe des canaux : 4 × 64 = 256 canaux
    return torch.cat(
      [
        self.conv11(entrees),
        self.conv33(entrees),
        self.conv55(entrees),
        self.pooling_max(entrees),
      ],
      dim=1,
    )


inception = nn.Sequential(
  BlocInception(FORME_ENTREE[0]),
  nn.Flatten(),
  nn.Linear(4 * 64 * 150 * 150, 6),
)
classifieur_inception, trainer_inception = entraine(
  inception, nb_epoques_max=10, nom="inception"
)

## Évaluation des performances sur l'ensemble de test

Le dossier `seg_test` contient 3 000 images que le modèle n'a vues ni à l'entraînement, ni en validation : sa justesse sur elles estime honnêtement celle qu'il aura sur de nouvelles photos. La méthode `test` du `Trainer` qui a entraîné le modèle la calcule. `justesses_test` garde la justesse en test de chaque approche, pour les comparer à la fin du TP.

In [ ]:
images_test, etiquettes_test = charge_images(pathlib.Path("landscape") / "seg_test")
chargeur_test = fabrique_chargeur(images_test, etiquettes_test, taille_lot=128)

# Le Trainer qui a entraîné le modèle est aussi celui qui l'évalue
resultats = trainer.test(classifieur, dataloaders=chargeur_test)
justesses_test = {"CNN profond, données augmentées": resultats[0]["justesse_test"]}


def predit_probabilites(
  trainer: lightning.Trainer,
  classifieur: ClassifieurImages,
  images: torch.Tensor,
  taille_lot: int = 128,
) -> torch.Tensor:
  """Renvoie la probabilité attribuée à chaque classe pour chaque image."""
  return torch.cat(
    trainer.predict(classifieur, fabrique_chargeur(images, taille_lot=taille_lot))
  )

## Analyse d'erreur

On affiche la matrice de confusion, puis des images mal classées.

In [ ]:
def analyse_predictions(
  indices_predits: torch.Tensor, etiquettes: torch.Tensor
) -> None:
  matrice_confusion = sklearn.metrics.confusion_matrix(etiquettes, indices_predits)
  seaborn.heatmap(
    matrice_confusion,
    annot=True,
    fmt="d",
    cmap="rocket_r",
    xticklabels=noms_etiquettes,
    yticklabels=noms_etiquettes,
  )
  plt.title("Matrice de confusion")
  plt.xlabel("Classe prédite")
  plt.ylabel("Classe réelle")
  plt.show()

  seaborn.countplot(
    x=[noms_etiquettes[indice_predit] for indice_predit in indices_predits.tolist()]
  )
  plt.title("Décomptes des classes prédites")
  plt.ylabel("Décompte")
  plt.xlabel("Classe")
  plt.show()


indices_predits_test = predit_probabilites(trainer, classifieur, images_test).argmax(
  dim=-1
)
analyse_predictions(indices_predits_test, etiquettes_test)

In [ ]:
def affiche_erreurs(classe_predite: str, classe_reelle: str) -> None:
  """Affiche jusqu'à 25 images de test de classe_reelle prédites classe_predite."""
  print(f"Prédiction : {classe_predite}, classe réelle : {classe_reelle}")
  erreurs = images_test[
    (indices_predits_test == noms_etiquettes.index(classe_predite))
    & (etiquettes_test == noms_etiquettes.index(classe_reelle))
  ]
  indices_aleatoires = numpy.random.choice(
    erreurs.shape[0], size=min(erreurs.shape[0], 25), replace=False
  )

  _, ax = plt.subplots(5, 5, figsize=(15, 15))
  indices_grille = list(itertools.product(range(5), repeat=2))
  for i, j in indices_grille:
    ax[i, j].axis("off")
  for indice_image, (i, j) in zip(indices_aleatoires, indices_grille):
    ax[i, j].imshow(rend_affichable(erreurs[indice_image]))
  plt.show()

In [ ]:
# Images prédites glacier alors qu'elles sont étiquetées montagne
affiche_erreurs("glacier", "mountain")

In [ ]:
# Images prédites glacier alors qu'elles sont étiquetées mer
affiche_erreurs("glacier", "sea")

In [ ]:
# Images prédites bâtiments alors qu'elles sont étiquetées rue
affiche_erreurs("buildings", "street")

## Transfert d'apprentissage

On réutilise un réseau pré-entraîné sur ImageNet, EfficientNetV2-B0 (bibliothèque `timm`), comme extracteur de caractéristiques : ses poids sont gelés, et on n'entraîne qu'une nouvelle tête de classification pour nos 6 classes (slides du chapitre « Transfer learning », dont ce TP reprend le code et les noms).

Les poids pré-entraînés attendent les images telles qu'à leur entraînement : `timm.data.resolve_model_data_config` donne leur taille, ainsi que la moyenne et l'écart type de leur normalisation. Ce réseau a été entraîné sur des images 192 × 192 : on recharge nos images à cette taille.

Avec `num_classes=0`, `timm` crée le réseau sans sa tête ImageNet (1 000 classes) : il renvoie, pour chaque image, la moyenne de ses 1 280 cartes de caractéristiques finales (pooling global).

In [ ]:
backbone = timm.create_model(
  "tf_efficientnetv2_b0.in1k", pretrained=True, num_classes=0
)
# Taille d'entrée, interpolation, moyenne et écart type de normalisation
config = timm.data.resolve_model_data_config(backbone)
print(config)

taille_transfert = config["input_size"][-1]
images, etiquettes = charge_images(
  pathlib.Path("landscape") / "seg_train", taille=taille_transfert
)
images_test, etiquettes_test = charge_images(
  pathlib.Path("landscape") / "seg_test", taille=taille_transfert
)
chargeur_test = fabrique_chargeur(images_test, etiquettes_test, taille_lot=128)

Comme dans les slides (« Prétraitements attendus par les poids »), la normalisation d'ImageNet, `v2.Normalize`, est la première couche du modèle : les CNN entraînés depuis zéro et le modèle pré-entraîné partagent ainsi les mêmes `DataLoader`, qui produisent des images dans [0, 1].

Le réseau pré-entraîné est encapsulé dans `Gele` (slide « Geler aussi les BatchNorm : `Frozen` »), qui gèle ses poids **et** le met en mode évaluation : ses BatchNorm normalisent avec les statistiques d'ImageNet sans les mettre à jour, et son dropout est éteint. `requires_grad=False` ne suffit pas : le mode d'un module ne dépend pas de ses gradients. Lightning garde le mode de chaque sous-module pendant `trainer.fit`, qui n'appelle pas `modele.train()` : la méthode `train` de `Gele`, qui refuse le mode entraînement, sert dans une boucle écrite à la main, qui l'appelle à chaque époque (chapitre PyTorch).

La nouvelle tête est une seule couche dense, des 1 280 caractéristiques vers nos 6 classes (slide « Remplacer la tête »). Elle apprend avec AdamW à un taux de 1e-3, comme dans les slides.

In [ ]:
# Normalisation d'ImageNet, canal par canal, d'images déjà ramenées dans [0, 1]
normalise = v2.Normalize(config["mean"], config["std"])


class Gele(nn.Module):
  """Module gardé en mode évaluation, même si l'entraînement appelle `train()`."""

  def __init__(self, module: nn.Module) -> None:
    super().__init__()
    self.module = module.eval().requires_grad_(False)

  def train(self, mode: bool = True) -> "Gele":
    # BatchNorm et dropout restent en mode évaluation
    return super().train(False)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.module(images)


modele = nn.Sequential(
  normalise,  # pixels dans [0, 1] → normalisés comme ImageNet
  Gele(backbone),  # 1 280 caractéristiques par image
  nn.Linear(backbone.num_features, 6),  # nouvelle tête
)

# La weight decay d'AdamW ne touche que les poids qui reçoivent un gradient :
# ceux de la tête, le réseau pré-entraîné étant gelé
classifieur = ClassifieurImages(
  modele,
  taux_apprentissage=1e-3,
  decroissance_poids=0.01,
  taille_entree=taille_transfert,
)

# max_depth=2 : sans le détail des couches d'EfficientNetV2
print(ModelSummary(classifieur, max_depth=2))

In [ ]:
chargeur_entrainement, chargeur_validation = fabrique_chargeurs(
  images, etiquettes, taille_lot=64
)
trainer = fabrique_trainer(nb_epoques_max=10, nom="transfert")
trainer.fit(classifieur, chargeur_entrainement, chargeur_validation)
trace_mesures(trainer)

In [ ]:
resultats = trainer.test(classifieur, dataloaders=chargeur_test)
justesses_test["EfficientNetV2-B0 gelé"] = resultats[0]["justesse_test"]

indices_predits_test = predit_probabilites(trainer, classifieur, images_test).argmax(
  dim=-1
)
analyse_predictions(indices_predits_test, etiquettes_test)
affiche_erreurs("glacier", "mountain")
affiche_erreurs("glacier", "sea")
affiche_erreurs("buildings", "street")

### Questions

1. Combien de paramètres ce modèle entraîne-t-il, et combien en gèle-t-il ? (voir son résumé)
2. Comparez sa justesse de test à celle du CNN profond entraîné depuis zéro, et la durée de leurs époques.
3. Les erreurs restantes (matrice de confusion, images mal classées) ressemblent-elles à celles du CNN ?

#### Solution

1. Seule la tête est entraînée : 1 280 × 6 + 6 = 7 686 paramètres, contre 5 858 704 gelés dans EfficientNetV2-B0.
2. Le modèle pré-entraîné dépasse le CNN profond dès les premières époques. Une époque ne coûte presque que le passage avant dans EfficientNetV2 : aucun gradient ne le traverse, puisqu'il est gelé. Les caractéristiques qu'il a apprises sur plus d'un million d'images d'ImageNet se transfèrent bien à nos 14 000 paysages.
3. Les confusions qui restent opposent surtout des classes réellement proches (glacier et montagne, bâtiments et rue) ; beaucoup de ces images sont ambiguës, voire mal annotées.

### Fine-tuning : dégeler le backbone

Une fois la tête entraînée, on peut dégeler EfficientNetV2 et entraîner tout le réseau quelques époques de plus, avec un taux d'apprentissage plus faible pour le backbone que pour la tête : ses poids sont déjà bons, on veut seulement les ajuster à nos paysages.

La cellule suivante dégèle le backbone comme dans les slides (« Fine-tuning : dégeler le backbone » et « Fine-tuning : garder les BatchNorm gelées ») : `Gele` retiré, poids dégelés et dropout actif, mais BatchNorm gelées, en mode évaluation. Avec 11 000 images en batchs de 64, les statistiques d'ImageNet sont plus fiables que celles des batchs ; Lightning, qui garde le mode de chaque sous-module, les laisse en évaluation pendant tout l'entraînement.

Il reste à donner au backbone son propre taux d'apprentissage. Écrivez `ClassifieurAffine`, un `ClassifieurImages` qui reçoit en plus `taux_apprentissage_backbone`, et dont la méthode `configure_optimizers` renvoie un AdamW à deux groupes de paramètres : le backbone (`self.modele[1]`) à `taux_apprentissage_backbone`, la tête (`self.modele[2]`) au taux `taux_apprentissage` de l'enveloppe, tous deux avec sa weight decay. Entraînez ensuite 3 époques, avec un taux de 1e-5 pour le backbone et de 1e-4 pour la tête, puis ajoutez la justesse en test à `justesses_test`. La justesse de validation progresse-t-elle ?

**Documentation utile** :

- https://lightning.ai/docs/pytorch/stable/common/lightning_module.html#configure-optimizers

In [ ]:
# Le backbone remplace son enveloppe Gele : poids dégelés, dropout actif
backbone = modele[1].module.train().requires_grad_(True)
modele[1] = backbone
# BatchNorm gelées : statistiques d'ImageNet conservées
for m in backbone.modules():
  if isinstance(m, nn.BatchNorm2d):
    m.eval().requires_grad_(False)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `ClassifieurAffine` hérite d'`ClassifieurImages` : son `__init__(self, modele, taux_apprentissage_backbone, **kwargs)` passe `modele` et les autres arguments à `super().__init__`, puis garde `taux_apprentissage_backbone` en attribut.
- `configure_optimizers` remplace celle d'`ClassifieurImages` (cellule « Création du modèle ») : `torch.optim.AdamW` reçoit une liste de groupes, des dictionnaires `{"params": ..., "lr": ...}`. Un groupe sans `"lr"` prend le taux passé à l'optimiseur, `lr=self.hparams.taux_apprentissage`, et `weight_decay=self.hparams.decroissance_poids` vaut pour tous.
- Les paramètres des BatchNorm, gelés, ne reçoivent pas de gradient : AdamW les laisse tels quels.
- `ClassifieurAffine(modele, taux_apprentissage_backbone=1e-5, taux_apprentissage=1e-4, decroissance_poids=0.01, taille_entree=taille_transfert)`, puis un nouveau `Trainer`, `fabrique_trainer(nb_epoques_max=3, nom="affinage")`, entraîné sur les mêmes `chargeur_entrainement` et `chargeur_validation`, puis `trace_mesures` et `trainer.test`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class ClassifieurAffine(ClassifieurImages):
  def __init__(
    self, modele: nn.Module, taux_apprentissage_backbone: float, **kwargs
  ) -> None:
    super().__init__(modele, **kwargs)
    self.taux_apprentissage_backbone = ...

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.AdamW(
      [
        {"params": self.modele[1].parameters(), "lr": ...},
        {"params": ...},
      ],
      lr=...,
      decroissance_poids=...,
    )


classifieur = ClassifieurAffine(
  modele,
  taux_apprentissage_backbone=...,
  taux_apprentissage=...,
  decroissance_poids=0.01,
  taille_entree=taille_transfert,
)
trainer = fabrique_trainer(nb_epoques_max=..., nom="affinage")
trainer.fit(...)
trace_mesures(trainer)
resultats = trainer.test(classifieur, dataloaders=chargeur_test)
justesses_test["EfficientNetV2-B0 fine-tuné"] = resultats[0]["justesse_test"]
```

#### Solution

In [ ]:
class ClassifieurAffine(ClassifieurImages):
  """ClassifieurImages dont le backbone (`modele[1]`) a son propre taux d'apprentissage."""

  def __init__(
    self, modele: nn.Module, taux_apprentissage_backbone: float, **kwargs
  ) -> None:
    super().__init__(modele, **kwargs)
    self.taux_apprentissage_backbone = taux_apprentissage_backbone

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.AdamW(
      [  # un taux par groupe de paramètres
        {"params": self.modele[1].parameters(), "lr": self.taux_apprentissage_backbone},
        {"params": self.modele[2].parameters()},  # la tête : le taux lr ci-dessous
      ],
      lr=self.hparams.taux_apprentissage,
      weight_decay=self.hparams.decroissance_poids,
    )


classifieur = ClassifieurAffine(
  modele,
  taux_apprentissage_backbone=1e-5,
  taux_apprentissage=1e-4,
  decroissance_poids=0.01,
  taille_entree=taille_transfert,
)
trainer = fabrique_trainer(nb_epoques_max=3, nom="affinage")
trainer.fit(classifieur, chargeur_entrainement, chargeur_validation)
trace_mesures(trainer)
resultats = trainer.test(classifieur, dataloaders=chargeur_test)
justesses_test["EfficientNetV2-B0 fine-tuné"] = resultats[0]["justesse_test"]

La justesse gagne encore un peu, quelques dixièmes de point ici : le backbone s'adapte à nos paysages. Un taux d'apprentissage trop élevé pour le backbone détruirait au contraire ce qu'il a appris sur ImageNet. La suite du TP utilise ce modèle.

## Prédire dans des conditions « réelles »

Le dossier `seg_pred` contient des images sans étiquette : on ne peut pas y mesurer la justesse. On peut en revanche afficher des photos et les probabilités que le modèle attribue à chaque classe.

In [ ]:
images_pred = charge_images(
  pathlib.Path("landscape") / "seg_pred", cree_etiquettes=False, taille=taille_transfert
)
print(f"{images_pred.shape[0]} images sans étiquette")

In [ ]:
# 25 images au hasard, sans remise
indices_aleatoires = numpy.random.choice(
  images_pred.shape[0], size=(5, 5), replace=False
)

# Probabilités des classes pour ces 25 images
probabilites = predit_probabilites(
  trainer, classifieur, images_pred[indices_aleatoires.ravel()]
).reshape(5, 5, -1)

# Chaque image, puis la distribution de ses prédictions sur la ligne d'en dessous
_, ax = plt.subplots(10, 5, figsize=(30, 45))
for i, j in itertools.product(range(5), repeat=2):
  indice_image = indices_aleatoires[i, j]
  classe_predite = noms_etiquettes[int(probabilites[i, j].argmax())]
  ax[i * 2, j].imshow(rend_affichable(images_pred[indice_image]))
  ax[i * 2, j].set_title(f"Image {indice_image} ({classe_predite})")
  ax[i * 2, j].axis("off")
  ax[i * 2 + 1, j].bar(noms_etiquettes, probabilites[i, j].numpy())
plt.show()

## Justesse en fonction de la probabilité maximale

La justesse varie-t-elle selon la probabilité maximale que rend le modèle ? Si oui, on peut ne garder que les prédictions sûres, et confier les autres à un humain.

In [ ]:
probabilites_test = predit_probabilites(trainer, classifieur, images_test)

In [ ]:
def justesse_seuil(
  probabilites: torch.Tensor, etiquettes: torch.Tensor, seuil: float
) -> tuple[float, float, float, float]:
  predictions = probabilites.argmax(dim=-1)
  masque_haut = probabilites.max(dim=-1).values > seuil
  masque_bas = ~masque_haut

  nb_haut = int(masque_haut.sum())
  nb_bas = int(masque_bas.sum())

  if nb_haut:
    haut = float((predictions[masque_haut] == etiquettes[masque_haut]).sum()) / nb_haut
  else:
    haut = 1.0
  if nb_bas:
    bas = float((predictions[masque_bas] == etiquettes[masque_bas]).sum()) / nb_bas
  else:
    bas = 1.0
  return haut, bas, nb_haut / etiquettes.shape[0], nb_bas / etiquettes.shape[0]


justesse_haut, justesse_bas, part_haut, part_bas = justesse_seuil(
  probabilites_test, etiquettes_test, 0.99
)
print(
  "Justesse pour les prédictions du modèle qui ont une probabilité au "
  f"dessus du seuil ({part_haut * 100:.2f}% des données) : "
  f"{justesse_haut:.2f}"
)
print(
  "Justesse pour les prédictions du modèle qui ont une probabilité en "
  f"dessous du seuil ({part_bas * 100:.2f}% des données) : "
  f"{justesse_bas:.2f}"
)

xs = numpy.arange(101) / 100
justesses = []
parts_haut = []
for x in xs:
  justesse, _, part_haut, _ = justesse_seuil(probabilites_test, etiquettes_test, x)
  justesses.append(justesse)
  parts_haut.append(part_haut)
plt.plot(xs, justesses, label="Justesse")
plt.plot(xs, parts_haut, label="Proportion de prédictions retenues")
plt.xlabel("Seuil de prédiction")
plt.title("Justesse et proportion de prédictions retenues selon le seuil")
plt.legend()
plt.show()

## Transfert d'apprentissage avec des transformeurs

Partie facultative : les slides présentent le *Vision Transformer* dans un chapitre à part (« Transformers »), s'il est au programme ; ce qu'il faut en savoir ici tient dans le paragraphe suivant.

Même démarche avec un *Vision Transformer* (ViT) pré-entraîné sur ImageNet-21k (14 millions d'images), chargé avec la bibliothèque `transformers`. Le ViT découpe une image 224 × 224 en 196 carrés de 16 × 16 pixels, et ajoute devant eux un token de classification, `[CLS]`. Le dernier état de ce token résume l'image : la tête de classification se branche sur lui.

La cellule suivante charge les prétraitements du ViT (`AutoImageProcessor` : taille, moyenne et écart type) et recharge les images à sa taille. Construisez ensuite `modele_vit`, un `nn.Sequential` :

- `v2.Normalize`, avec la moyenne et l'écart type du ViT ;
- le ViT (`transformers.AutoModel`), gelé par `Gele`, derrière un module `CaracteristiquesViT` à écrire, qui renvoie l'état final du token `[CLS]` ;
- une couche dense de sortie, de la taille de cet état (768) vers nos 6 classes.

Enveloppez-le dans un `ClassifieurImages` nommé `classifieur`, avec un taux d'apprentissage de 1e-3, comme la tête d'EfficientNetV2.

**Documentation utile** :

- https://huggingface.co/docs/transformers/model_doc/vit#transformers.ViTModel

In [ ]:
VIT = "google/vit-base-patch16-224-in21k"
# Prétraitements attendus par le ViT : taille, moyenne et écart type
pretraitement = transformers.AutoImageProcessor.from_pretrained(VIT)
taille_vit = pretraitement.size["height"]
print(taille_vit, pretraitement.image_mean, pretraitement.image_std)

images, etiquettes = charge_images(
  pathlib.Path("landscape") / "seg_train", taille=taille_vit
)
images_test, etiquettes_test = charge_images(
  pathlib.Path("landscape") / "seg_test", taille=taille_vit
)
chargeur_test = fabrique_chargeur(images_test, etiquettes_test, taille_lot=128)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Chargement : `transformers.AutoModel.from_pretrained(VIT)` (350 Mo à télécharger).
- `CaracteristiquesViT` est un `nn.Module` qui garde le ViT en attribut. Sa méthode `forward` appelle `self.vit(pixel_values=images)`, dont l'attribut `last_hidden_state` donne un vecteur de 768 valeurs par token, de forme `(N, 197, 768)` : le token `[CLS]` est le premier, `[:, 0]`.
- `v2.Normalize(pretraitement.image_mean, pretraitement.image_std)` ramène les pixels de [0, 1] à [-1, 1].
- La taille de l'état du token vaut `vit.config.hidden_size`.
- `ClassifieurImages(modele_vit, taux_apprentissage=1e-3, taille_entree=taille_vit)`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class CaracteristiquesViT(nn.Module):
  def __init__(self, vit: nn.Module) -> None:
    super().__init__()
    self.vit = vit

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return ...


vit = transformers.AutoModel.from_pretrained(VIT)
modele_vit = nn.Sequential(
  v2.Normalize(...),
  Gele(...),
  nn.Linear(...),
)
classifieur = ClassifieurImages(...)
print(ModelSummary(classifieur, max_depth=2))
```

### Solution

In [ ]:
class CaracteristiquesViT(nn.Module):
  """Renvoie l'état final du token de classification [CLS] calculé par le ViT."""

  def __init__(self, vit: nn.Module) -> None:
    super().__init__()
    self.vit = vit

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    # (N, 197, 768) : un vecteur par token, [CLS] en premier
    return self.vit(pixel_values=images).last_hidden_state[:, 0]


vit = transformers.AutoModel.from_pretrained(VIT)

modele_vit = nn.Sequential(
  v2.Normalize(pretraitement.image_mean, pretraitement.image_std),  # [0, 1] → [-1, 1]
  Gele(CaracteristiquesViT(vit)),
  nn.Linear(vit.config.hidden_size, 6),
)
classifieur = ClassifieurImages(
  modele_vit, taux_apprentissage=1e-3, taille_entree=taille_vit
)
print(ModelSummary(classifieur, max_depth=2))

### Comparaison des trois approches

Entraînez le ViT 5 époques, évaluez-le sur l'ensemble de test, puis comparez les trois approches : CNN entraîné depuis zéro, EfficientNetV2-B0 pré-entraîné, ViT pré-entraîné. Qu'apporte le transfert d'apprentissage, et à quel prix ?

In [ ]:
chargeur_entrainement, chargeur_validation = fabrique_chargeurs(
  images, etiquettes, taille_lot=64
)
trainer = fabrique_trainer(nb_epoques_max=5, nom="vit")
trainer.fit(classifieur, chargeur_entrainement, chargeur_validation)
trace_mesures(trainer)

In [ ]:
resultats = trainer.test(classifieur, dataloaders=chargeur_test)
justesses_test["ViT gelé"] = resultats[0]["justesse_test"]

indices_predits_test = predit_probabilites(trainer, classifieur, images_test).argmax(
  dim=-1
)
analyse_predictions(indices_predits_test, etiquettes_test)

for nom, justesse in justesses_test.items():
  print(f"{nom} : {justesse:.3f}")

#### Solution

- Les deux modèles pré-entraînés dépassent nettement le CNN entraîné depuis zéro (93 à 94 % en test, contre 84 %), en 5 à 13 époques au lieu de 15 : c'est tout l'intérêt du transfert d'apprentissage quand on a peu de données (14 000 images ici, contre plus d'un million pour ImageNet et 14 millions pour ImageNet-21k).
- Entre EfficientNetV2-B0 et le ViT, l'écart est faible, mais le ViT est bien plus gros : 86 millions de paramètres contre 5,9 millions, et chaque prédiction coûte en proportion, comme son entraînement : ses 5 époques durent autant que les 15 du CNN. Le fine-tuning du backbone apporte en général les derniers points.
- Les confusions restantes sont les mêmes pour les trois modèles : glacier et montagne, bâtiments et rue.